# Mejoras Thyroid — Guía de Examen

## Tipo de problema
**Clasificación supervisada multiclase** con **desbalance de clases** y contexto médico.

Clases:
- `negative` (~92%) — paciente sano
- `hypothyroid` (~7%) — tiroides poco activa
- `hyperthyroid` (~1%) — tiroides demasiado activa

## Por qué este problema es diferente a regresión

En regresión queremos minimizar el error numérico.
En clasificación médica queremos **minimizar falsos negativos** — decirle a un enfermo que está sano
es mucho peor que decirle a un sano que quizá deba hacerse más pruebas.

## Mejoras clave
1. **F2-score** en lugar de accuracy (penaliza más los falsos negativos)
2. **`class_weight="balanced"`** para que el modelo no ignore las clases minoritarias
3. **Simplificación de clases** (los originales tienen 30 subclases, las reducimos a 3)
4. **Manejo correcto de valores imposibles** (edades > 150 → NaN, no eliminar la fila)
5. **Stratify en el split** para mantener la proporción de clases en train y test


## Paso 1 — Cargar y simplificar clases

**Por qué simplificar de 30 subclases a 3?**
El dataset original tiene clases como "A", "AK", "B", "C", "D", "E"...
Algunos subtipos tienen 5 muestras en todo el dataset — imposible entrenar con eso.
Las agrupamos según el código médico: A-D = hipertiroidismo, E-H = hipotiroidismo.

**Por qué no eliminar filas con `target` NaN?**
Solo eliminamos filas donde el **target** es NaN (no podemos entrenar sin etiqueta).
Las filas con NaN en features sí las mantenemos — el pipeline los imputará.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("datasets/thyroid/thyroidDF.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../proyects-upgrades/datasets/thyroid/thyroidDF.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Falta thyroidDF.csv. Descárgalo con:\n"
        "  kagglehub.dataset_download('emmanuelfwerr/thyroid-disease-data')"
    )

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print("Clases originales (primeras 10):")
print(df["target"].value_counts().head(10))


In [ ]:
def simplify_thyroid_class(value: str) -> str:
    """
    Mapea los 30 subtipos médicos a 3 clases principales.
    Códigos A-D: hipertiroidismo (tiroides hiperactiva)
    Códigos E-H: hipotiroidismo (tiroides hipoactiva)
    El resto: negativo (sano)
    """
    if any(code in value for code in ["A", "B", "C", "D"]):
        return "hyperthyroid"
    if any(code in value for code in ["E", "F", "G", "H"]):
        return "hypothyroid"
    return "negative"

df["target"] = df["target"].apply(simplify_thyroid_class)
df = df.dropna(subset=["target"])  # solo eliminar si no tenemos etiqueta

print("Clases simplificadas:")
print(df["target"].value_counts())
print()
print(f"Desbalance: {df['target'].value_counts(normalize=True).round(3).to_string()}")


## Paso 2 — Preparar features y manejar valores imposibles

**Por qué tratar edad > 150 como NaN en vez de eliminar la fila?**
Una edad de 999 es claramente un error de captura de datos.
Si eliminamos la fila, perdemos toda la información del paciente (sus niveles de TSH, T3, T4...).
Si la reemplazamos por NaN, el imputador la rellena con la mediana de edades válidas,
preservando el resto de la información.

**Por qué eliminar `patient_id` y `referral_source`?**
- `patient_id`: identificador único del paciente, no contiene información médica
- `referral_source`: indica de dónde vino el paciente (hospital, médico de familia...).
  Puede introducir sesgo si el hospital A siempre manda casos más graves.
  En producción habría que analizarlo; en examen es más seguro eliminarlo.


In [ ]:
# Separar features y target
X = df.drop(columns=["patient_id", "referral_source", "target"])
y = df["target"]

# Tratar edades imposibles como datos faltantes (no eliminar la fila)
X = X.copy()
X.loc[X["age"] > 150, "age"] = np.nan

print(f"Features: {X.shape[1]} columnas")
print(f"NaN por columna (primeras con NaN):")
nan_counts = X.isnull().sum()
print(nan_counts[nan_counts > 0].sort_values(ascending=False))


## Paso 3 — Split estratificado

**Por qué `stratify=y` es OBLIGATORIO en clasificación desbalanceada?**
Con 92% negativos, un split aleatorio podría dar un test con 95% negativos y 5% enfermos,
o 89% negativos y 11% enfermos. Nuestras métricas variarían mucho dependiendo del azar.
`stratify=y` garantiza que la proporción de cada clase sea igual en train y test.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,     # ← obligatorio con clases desbalanceadas
)

print("Distribución de clases en TRAIN:")
print(y_train.value_counts(normalize=True).round(3))
print("\nDistribución de clases en TEST:")
print(y_test.value_counts(normalize=True).round(3))
print("\n✓ Las proporciones deben ser similares en ambos splits")


## Paso 4 — Métrica F2

**Por qué F2 y no accuracy, F1 o precision?**

| Métrica | Qué maximiza | Problema en medicina |
|---------|-------------|----------------------|
| Accuracy | Clasificaciones correctas totales | Con 92% negativos, un modelo "siempre negativo" tiene 92% accuracy |
| Precision | Evitar falsos positivos | Prefiere no diagnosticar aunque deje enfermos sin tratar |
| Recall | Evitar falsos negativos | Ningún tradeoff con precision |
| F1 | Balance igual entre precision y recall | Trata igual el error de decir "sano" al enfermo que "enfermo" al sano |
| **F2** | **Recall dos veces más importante que precision** | **Detectar enfermos es más urgente que no alarmar a sanos** |

**Fórmula F-beta:**
```
F_beta = (1 + beta²) × (precision × recall) / (beta² × precision + recall)
```
Con `beta=2`: el recall tiene el doble de peso que la precision.

**Por qué `labels=["hyperthyroid", "hypothyroid"]` y no incluir "negative"?**
Nos interesa detectar las clases enfermas. Si incluimos "negative", el modelo podría
compensar un mal recall en enfermos con un recall perfecto en sanos.


In [ ]:
from sklearn.metrics import fbeta_score, make_scorer

def thyroid_f2(y_true, y_pred):
    """
    F2 macro sobre solo las clases enfermas.
    beta=2 → recall cuenta el doble que precision.
    average='macro' → media simple de F2 de hyper e hypo (sin pesar por frecuencia).
    zero_division=0 → si una clase no aparece en predicciones, F2=0 (no error)
    """
    return fbeta_score(
        y_true,
        y_pred,
        beta=2,
        labels=["hyperthyroid", "hypothyroid"],
        average="macro",
        zero_division=0,
    )

# Envolver en make_scorer para usarlo en GridSearch
# greater_is_better=True porque F2 más alto = mejor
thyroid_scorer = make_scorer(thyroid_f2, greater_is_better=True)

print("Scorer creado. Se usará en RandomizedSearchCV como scoring=thyroid_scorer")


## Paso 5 — Pipeline y búsqueda de hiperparámetros

**Por qué `class_weight="balanced"` en RandomForest?**
Sin este parámetro, el árbol ve 92% negativos en cada nodo y aprende a predecir "negative" para todo.
Con `class_weight="balanced"`, sklearn pesa cada muestra inversamente proporcional a la frecuencia de su clase:
- `negative`: peso bajo (hay muchos)
- `hypothyroid`: peso medio
- `hyperthyroid`: peso alto (hay muy pocos)

Así el modelo presta más atención a los casos raros.

**Por qué numéricas y categóricas tienen pipelines distintos?**
Las columnas booleanas del thyroid dataset (0/1) son técnicamente numéricas pero
tienen muy pocos valores únicos. El imputador de mediana funciona bien para ambas.
Las pocas columnas de texto (como `sex`) necesitan one-hot encoding.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols     = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),   # NaN → mediana
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),  # NaN → moda
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",    # ← clave para datos desbalanceados
        n_jobs=-1,
    )),
])

param_grid = {
    "classifier__n_estimators":     [200, 400],
    "classifier__max_depth":        [8, 16, None],
    "classifier__min_samples_leaf": [1, 3, 5],
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=8,                     # 8 combinaciones aleatorias
    scoring=thyroid_scorer,       # ← optimizar F2 sobre clases enfermas
    cv=5,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train)
print("Mejores parámetros:", search.best_params_)
print(f"Mejor F2 CV: {search.best_score_:.4f}")


In [ ]:
y_pred = search.best_estimator_.predict(X_test)

print("=" * 50)
print("CLASSIFICATION REPORT (TEST SET)")
print("=" * 50)
print(classification_report(y_test, y_pred, zero_division=0))

print(f"\nThyroid F2 (métrica objetivo): {thyroid_f2(y_test, y_pred):.4f}")
print("(Objetivo: F2 > 0.80 sobre clases enfermas)")


## Explicación para el examen

> *"Es un problema de clasificación médica con clases muy desbalanceadas: el 92% de pacientes son sanos.
> Accuracy es una métrica trampa aquí — un modelo que siempre predice 'negativo' tiene 92% accuracy.
> Optimizo F2-score sobre las clases enfermas, que da el doble de peso al recall que a la precision,
> porque diagnosticar mal a un enfermo es peor que alarmar innecesariamente a un sano.
> Uso class_weight='balanced' para que RandomForest no ignore las clases minoritarias.
> Las edades imposibles se convierten en NaN y el imputador las trata, sin perder la fila completa."*

## Problemas frecuentes y soluciones

| Problema | Causa | Solución |
|----------|-------|---------|
| F2 = 0.0 | Modelo predice siempre "negative" | Añadir `class_weight="balanced"` |
| Accuracy 92% pero F2 bajo | Modelo ignora clases raras | Igual que arriba |
| `fbeta_score` da warning | Una clase no aparece en predicciones | Añadir `zero_division=0` |
| Error en `make_scorer` | `greater_is_better` no especificado | Añadir explícitamente |
| Clases con 0 muestras en test | Split no estratificado | Usar `stratify=y` en `train_test_split` |
| NaN en toda una columna | Solo hay valores imposibles | Verificar los datos antes de imputar |
